# Lab #17 — Multi-Memory Agent demo

Interactive walkthrough of the LangGraph agent with 4 memory backends.

Covers:
1. Build the agent + seed semantic memory
2. Inspect the graph flow
3. Run a profile recall scenario
4. Run the mandatory conflict update test
5. Inspect all 4 memory stores
6. Trim/budget sanity check

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from dotenv import load_dotenv
load_dotenv()

from agent import MemoryAgent
from benchmark.seed_semantic import seed

agent = MemoryAgent()
agent.reset_memories()
seed(agent.semantic)
print(f'Semantic chunks seeded: {agent.semantic.count()}')

## 2. Graph structure

The compiled LangGraph has four real nodes wired linearly from entry to END.

In [ ]:
print(agent.graph.get_graph().draw_ascii())

## 3. Profile recall after a few turns

In [ ]:
for turn in [
    'Xin chào, tôi tên là Linh.',
    'Tôi đang học về agent memory.',
    'Cảm ơn, nghe có vẻ thú vị.',
    'Nhắc lại giúp tôi: tên tôi là gì?',
]:
    out = agent.chat(turn)
    print(f'>>> {turn}')
    print(f'... {out["response"]}')
    print(f'    (prompt_tokens={out["prompt_tokens"]})\n')
print('Final profile:', agent.profile.all())

## 4. Mandatory conflict update test

> User: Tôi dị ứng sữa bò.
>
> User: À nhầm, tôi dị ứng đậu nành chứ không phải sữa bò.
>
> Expected: allergy = đậu nành

In [ ]:
agent.reset_memories()
seed(agent.semantic)

for turn in [
    'Tôi tên Mai, tôi dị ứng sữa bò.',
    'À nhầm, tôi dị ứng đậu nành chứ không phải sữa bò.',
    'Tôi dị ứng gì nhỉ?',
]:
    out = agent.chat(turn)
    print(f'>>> {turn}')
    print(f'... {out["response"]}\n')

print('Profile:', agent.profile.all())
print('\nAudit history for allergy:')
for h in agent.profile.history('allergy'):
    print(' ', h)

## 5. Inspect all 4 memory stores

In [ ]:
print('--- Short-term (sliding window) ---')
print(agent.short.as_text())

print('\n--- Profile ---')
print(agent.profile.as_text())

print('\n--- Episodic (last 3) ---')
print(agent.episodic.as_text(agent.episodic.recent(3)))

print('\n--- Semantic top-3 for "policy" ---')
hits = agent.semantic.search('policy', k=3)
for h in hits:
    print(f"  ({h['metadata'].get('topic','?')}) d={h['distance']:.3f} :: {h['text'][:90]}...")

## 6. Token budget: compare memory-on vs memory-off on the same question

In [ ]:
q = 'Tôi dị ứng gì nhỉ?'
off = agent.chat(q, with_memory=False)
on  = agent.chat(q, with_memory=True)
print('no-memory  :', off['prompt_tokens'], 'tokens ::', off['response'])
print('with-memory:', on['prompt_tokens'], 'tokens ::', on['response'])